In [5]:
import os
from dotenv import load_dotenv
from openrouter import OpenRouter
from langchain_community.vectorstores import Chroma

load_dotenv()
api_key=os.getenv("OPENROUTER_API_KEY")

SYSTEM_PROMPT = """Role:
    คุณเป็นผู้เชี่ยวชาญด้านการวิเคราะห์ SMS หลอกลวง ในประเทศไทย และเป็นหน่วยรักษาความปลอดภัยทางไซเบอร์มานานกว่า 50 ปี

    Task:
    โมเดล NLP ตรวจพบว่าข้อความนี้มีแนวโน้มเป็นข้อความหลอกลวง
    หน้าที่ของคุณคือวิเคราะห์และอธิบายเหตุผลว่าทำไมข้อความนี้จึงเป็น SMS scam

    Analysis method:
    1. ระบุจุดน่าสงสัยในข้อความ เช่น การส่งลิงก์ปลอม, การแอบอ้าง, การให้ข้อมูลเท็จ
    2. อ้างอิงนโยบายจริงขององค์กรหรือธนาคารที่เกี่ยวข้องจาก Context ที่ให้มา
    3. แสดงให้เห็นถึงความขัดแย้งระหว่างข้อความกับนโยบายจริง

    Response format:
    ตอบเป็นภาษาไทยเท่านั้นห้ามตอบเป็นภาษอื่นเด็ดขาด กระชับ เข้าใจง่าย และใช้โครงสร้างตามดังนี้:
    - จุดที่น่าสงสัย (ระบุจุดน่าสงสัยที่พบ)
    - นโยบายที่เกี่ยวข้อง (อ้างอิงจาก Context)
    - สรุป (อธิบายสั้นๆ ว่าทำไมถึงเป็น SMS Scam)

    Important:
    - ให้ใช้เฉพาะข้อมูลจาก Context ที่ให้มาเท่านั้น ห้ามแต่งนโยบายขึ้นมาเอง
    - ถ้าไม่มีนโยบายที่เกี่ยวข้องใน Context ให้วิเคราะห์จากลักษณะของข้อความแทน
    """

USER_PROMPT_TEMPLATE = """ข้อความ SMS: {sms_message}
นโยบายอ้างอิง:
{retrieved_context}"""

def analyze_scam_sms(sms_message: str, retriever, api_key: str = None) -> str:

    retrieved_docs = retriever.invoke(sms_message)

    context = "\n\n---\n\n".join([
        f"[ที่มา: {doc.metadata.get('source', 'ระบุแหล่งที่มา')}]\n{doc.page_content}"
        for doc in retrieved_docs
    ])

    user_prompt = USER_PROMPT_TEMPLATE.format(
        sms_message=sms_message,
        retrieved_context=context
    )

    with OpenRouter(api_key) as client:
        response = client.chat.send(
            model="qwen/qwen3-32b",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ]
        )

    return response.choices[0].message.content


In [8]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)

#load from chroma
persist_directory = "./chroma_db"
vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embedding_model
)

retriever = vectorstore.as_retriever(
    search_type="similarity",           
    search_kwargs={"k": 3}
)

sms_message = "คุณได้รับสินเชื่อ BAAC ธกส. 80,000 บาท สนใจคลิก :  bit.ly/3h9V64V,1,Financial/Bank"

print(f"\nข้อความ SMS: {sms_message}")

result = analyze_scam_sms(sms_message=sms_message, retriever=retriever, api_key=api_key)

print("===  จากการวิเคราห์ของ AI จะสรุปได้ว่า ===")
print(result)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5324.35it/s]



ข้อความ SMS: คุณได้รับสินเชื่อ BAAC ธกส. 80,000 บาท สนใจคลิก :  bit.ly/3h9V64V,1,Financial/Bank
===  จากการวิเคราห์ของ AI จะสรุปได้ว่า ===
- **จุดที่น่าสงสัย**  
  ข้อความแจ้งว่าผู้รับได้รับสินเชื่อจาก BAAC ธกส. โดยมีลิงก์ให้คลิกลงทะเบียน ซึ่งเป็นการเสนอเงินกู้โดยไม่ทราบรายละเอียดการสมัครรับสิทธิ์ล่วงหน้าหรือการตรวจสอบเครดิต ลักษณะนี้มักเป็นวิธีนิยมของผู้หลอกลวงให้คลิกลิงก์ปลอมเพื่อฉกข้อมูลส่วนบุคคล

- **นโยบายที่เกี่ยวข้อง**  
  ธกส. (ธนาคารเพื่อการเกษตรและสหกรณ์การเกษตร) มีนโยบายแจ้งข้อมูลหรือเสนอสินเชื่อต่าง ๆ โดยผ่านช่องทางการรับข้อมูลอย่างเป็นทางการ เช่น การโทรโดยผู้ให้บริการติดต่อโดยตรง, การส่งผ่านเว็บไซต์ทางการ, Mobile Application "BAAC LINE" และไม่เคยขอข้อมูลส่วนบุคคลผ่านลิงก์สั้นผิดปกติหรือ SMS

- **สรุป**  
  ข้อความนี้มีลักษณะเป็น SMS scam เนื่องจากเสนอสินเชื่อโดยไม่แจ้งข้อมูลชัดเจน และมีการส่งลิงก์สั้นที่ไม่สามารถตรวจสอบความน่าเชื่อถือได้ ขัดกับนโยบายการให้บริการและความเป็นจริงของธกส.
